In [1]:
# Clear all variables
%reset -f
%cd /home/zwei1/Artem/git_book/handbook_chapter/sfa-python/sfa/inefficiency_predictors/

/home/zwei1/Artem/git_book/handbook_chapter/sfa-python/sfa/inefficiency_predictors


In [2]:
from util import (
    JLMS_panel_technical_inefficiency_scores,
    NW_panel_technical_inefficiency_scores,
    inverse_mapping_vec,
 )
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
from scipy import stats

In [3]:
#load parameters


theta = np.array([
    5.7630,
    0.1422,
    0.1170,
    0.0720,
    0.1851,
    0.5150,
    -0.0191,
    0.1561,
    0.0996,
    -0.0355,
    -0.1337,
    -0.1617,
    -0.1609,
    -0.0293,
    0.6475,
    2.0451
])

print(theta)

gamma = np.array([
4.691359544478445054e-01,
2.449463444476085439e-01,
4.738861695923803768e-02,
1.416673024522306368e-01,
2.191707371168801211e-01,
9.297518610871904343e-01,
7.927239502606412413e-01,
1.393802658718614085e+00,
3.252880911850812573e-02,
-4.430338547558972384e-01,
-9.527848585280578320e-01,
1.232980848191408789e-02,
-1.114131598725027317e+00,
2.370215935915223338e-01,
7.045769769525236370e-01
])
gamma

[ 5.763   0.1422  0.117   0.072   0.1851  0.515  -0.0191  0.1561  0.0996
 -0.0355 -0.1337 -0.1617 -0.1609 -0.0293  0.6475  2.0451]


array([ 0.46913595,  0.24494634,  0.04738862,  0.1416673 ,  0.21917074,
        0.92975186,  0.79272395,  1.39380266,  0.03252881, -0.44303385,
       -0.95278486,  0.01232981, -1.1141316 ,  0.23702159,  0.70457698])

In [4]:
ricefarm = pd.read_csv(r"ricefarm.csv")

ricefarm = ricefarm.iloc[:, [0, 1, 3, 5, 6, 7, 8, 14, 16]]

# Create ID dummies
dar = np.round(ricefarm["ID"] / 100000).astype(int)
id_dummies = pd.get_dummies(dar, prefix="DR").astype(int)
ricefarm = pd.concat([ricefarm, id_dummies.iloc[:, :-1]], axis=1)

# Create rice variety dummies
rice_dummies = pd.get_dummies(ricefarm.iloc[:, 2], prefix="VAR").astype(int)
ricefarm = pd.concat([ricefarm, rice_dummies.iloc[:, 1:]], axis=1)

# Recode TSP as logs and zeros
ricefarm[ricefarm.columns[5]] = np.where(
    ricefarm.iloc[:, 5] > 0, np.log(ricefarm.iloc[:, 5]), 0
)

# Convert pesticide usage to a dummy
ricefarm[ricefarm.columns[6]] = (ricefarm.iloc[:, 6] != 0).astype(int)

# Reorder the data
ricefarm = ricefarm.iloc[:, [8, 3, 4, 5, 7, 1, 6, 14, 15, 9, 10, 11, 12, 13]]

y = np.log(ricefarm.iloc[:, 0].values)
X = ricefarm.iloc[:, 1:].values

# Log covariates
X[:, 0:2] = np.log(X[:, 0:2])
X[:, 3] = np.log(X[:, 3])
X[:, 4] = np.log(X[:, 4])

# Cross sections of  y and X
N = 171
T = 6
y_t = []
X_t = []
for t in range(T):
    if t == 0:
        y_t.append(y[:N])
        X_t.append(X[:N, :])
    else:
        y_t.append(y[t * N : (t + 1) * N])
        X_t.append(X[t * N : (t + 1) * N, :])

# Import the model parameters from exercise 5.8
# theta = np.loadtxt("exercise_4_8_theta_python.txt", delimiter=",")

# Estimation of technical inefficiencies
# Get idx of percentile firms
percetile_firm_idx = np.round(np.arange(0.05, 1, 0.1) * N).astype(int)

# JLMS formula technical inefficiencies
JLMS_u_hat = JLMS_panel_technical_inefficiency_scores(theta=theta, y=y_t, X=X_t)
# Sort technical inefficiency scores by t=1 values
sort_idx = JLMS_u_hat[:, 0].argsort()
sorted_t1_JLMS_u_hat = JLMS_u_hat[sort_idx]
# Mean technical inefficiency for each firms over all t
JLMS_mean = np.mean(JLMS_u_hat, axis=1)
sorted_t1_JLMS_mean = JLMS_mean[sort_idx]
# Standard deviation of technical inefficiency for each firm over all t
JLMS_std = np.std(JLMS_u_hat, axis=1)
sorted_t1_JLMS_std = JLMS_std[sort_idx]

JLMS_u_hat_for_table = sorted_t1_JLMS_u_hat[percetile_firm_idx-1, :]
JLMS_std_u_hat_for_table = sorted_t1_JLMS_std[percetile_firm_idx-1]
JLMS_mean_u_hat_for_table = sorted_t1_JLMS_mean[percetile_firm_idx-1]

# Tabulate the technical inefficiency scores
JLMS_technical_inefficiency_results = pd.DataFrame(
    columns=[
        "Rank",
        "Fractile",
        "Std(u)",
        "Mean(u)",
        "t=1",
        "t=2",
        "t=3",
        "t=4",
        "t=5",
        "t=6",
    ]
)
JLMS_technical_inefficiency_results["Rank"] = percetile_firm_idx
JLMS_technical_inefficiency_results["Fractile"] = np.arange(0.05, 1, 0.1)
JLMS_technical_inefficiency_results["Std(u)"] = JLMS_std_u_hat_for_table
JLMS_technical_inefficiency_results["Mean(u)"] = JLMS_mean_u_hat_for_table
JLMS_technical_inefficiency_results.iloc[:, -6:] = JLMS_u_hat_for_table

# Compute averages
average_JLMS = pd.DataFrame(
    np.concatenate(
        [
            np.mean(JLMS_std_u_hat_for_table).reshape(-1, 1).T,
            np.mean(JLMS_mean_u_hat_for_table).reshape(-1, 1).T,
            np.mean(JLMS_u_hat_for_table, axis=0).reshape(-1, 1).T,
        ],
        axis=1,
    ),
    columns=["Std(u)", "Mean(u)", "t=1", "t=2", "t=3", "t=4", "t=5", "t=6"],
    index=["Average"],
)

print("JLMS Scores")
display(JLMS_technical_inefficiency_results)
display(average_JLMS)

JLMS Scores


/home/zwei1/.conda/envs/d2l_f25/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Rank,Fractile,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
0,9,0.05,0.199552,0.353685,0.155262,0.334548,0.751723,0.298472,0.168863,0.413244
1,26,0.15,0.266314,0.389385,0.19444,0.247792,0.901213,0.577862,0.251281,0.163724
2,43,0.25,0.089338,0.285361,0.22775,0.192722,0.460339,0.332544,0.232714,0.266099
3,60,0.35,0.139117,0.389762,0.258728,0.210491,0.295984,0.535175,0.477761,0.560436
4,77,0.45,0.200050,0.338565,0.283924,0.201444,0.559467,0.664157,0.178611,0.143785
5,94,0.55,0.065625,0.362564,0.307167,0.346151,0.296446,0.381004,0.349333,0.495284
6,111,0.65,0.108977,0.455180,0.336395,0.376161,0.39316,0.559239,0.642827,0.423299
7,128,0.75,0.139393,0.307534,0.393146,0.551261,0.307115,0.291799,0.134207,0.167675
8,145,0.85,0.293302,0.600276,0.466422,0.382638,1.180244,0.717366,0.284154,0.570831
9,162,0.95,0.138611,0.552418,0.568983,0.719867,0.540943,0.716203,0.425787,0.342725


,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
Average,0.164028,0.403473,0.319222,0.356307,0.568664,0.507382,0.314554,0.35471


In [5]:
ricefarm = pd.read_csv(r"ricefarm.csv")

ricefarm = ricefarm.iloc[:, [0, 1, 3, 5, 6, 7, 8, 14, 16]]

# Create ID dummies
dar = np.round(ricefarm["ID"] / 100000).astype(int)
id_dummies = pd.get_dummies(dar, prefix="DR").astype(int)
ricefarm = pd.concat([ricefarm, id_dummies.iloc[:, :-1]], axis=1)

# Create rice variety dummies
rice_dummies = pd.get_dummies(ricefarm.iloc[:, 2], prefix="VAR").astype(int)
ricefarm = pd.concat([ricefarm, rice_dummies.iloc[:, 1:]], axis=1)

# Recode TSP as logs and zeros
ricefarm[ricefarm.columns[5]] = np.where(
    ricefarm.iloc[:, 5] > 0, np.log(ricefarm.iloc[:, 5]), 0
)

# Convert pesticide usage to a dummy
ricefarm[ricefarm.columns[6]] = (ricefarm.iloc[:, 6] != 0).astype(int)

# Reorder the data
ricefarm = ricefarm.iloc[:, [8, 3, 4, 5, 7, 1, 6, 14, 15, 9, 10, 11, 12, 13]]

y = np.log(ricefarm.iloc[:, 0].values)
X = ricefarm.iloc[:, 1:].values

# Log covariates
X[:, 0:2] = np.log(X[:, 0:2])
X[:, 3] = np.log(X[:, 3])
X[:, 4] = np.log(X[:, 4])

# Cross sections of  y and X
N = 171
T = 6
y_t = []
X_t = []
for t in range(T):
    if t == 0:
        y_t.append(y[:N])
        X_t.append(X[:N, :])
    else:
        y_t.append(y[t * N : (t + 1) * N])
        X_t.append(X[t * N : (t + 1) * N, :])
        
# Import the model parameters from exercise 5.8
# theta = np.loadtxt("exercise_4_8_theta_python.txt", delimiter=",")
# gamma = np.loadtxt("exercise_4_8_gamma_python.txt", delimiter=",")

# Estimation of technical inefficiencies
# Get idx of percentile firms
percetile_firm_idx = np.round(np.arange(0.05, 1, 0.1) * N).astype(int)

# NW technical inefficiency estimates
Rho_hat = inverse_mapping_vec(gamma)
# Draw T dependent unfirm RV's for each s_{1}, ... 10000
Z = stats.multivariate_normal.rvs(
    size=10000, mean=np.zeros(T), cov=np.eye(T), random_state=1234
)
A = np.linalg.cholesky(Rho_hat)
U_hat = stats.norm.cdf(
    Z @ A, loc=np.zeros(T), scale=np.ones(T)
)  # Dependent uniform RVs from a gaussian copula
NW_u_hat = NW_panel_technical_inefficiency_scores(
    theta=theta, y=y_t, X=X_t, U_hat=U_hat
)

sort_idx = NW_u_hat[:, 0].argsort()
# Sort technical inefficiency scores by t=1 values
sorted_t1_NW_u_hat = NW_u_hat[sort_idx]
# Mean technical inefficiency for each firms over all t
NW_mean = np.mean(NW_u_hat, axis=1)
sorted_t1_NW_mean = NW_mean[sort_idx]
# Standard deviation of technical inefficiency for each firm over all t
NW_std = np.std(NW_u_hat, axis=1)
sorted_t1_NW_std = NW_std[sort_idx]

NW_u_hat_for_table = sorted_t1_NW_u_hat[percetile_firm_idx-1, :]
NW_std_u_hat_for_table = sorted_t1_NW_std[percetile_firm_idx-1]
NW_mean_u_hat_for_table = sorted_t1_NW_mean[percetile_firm_idx-1]

NW_technical_inefficiency_results = pd.DataFrame(
    columns=[
        "Rank",
        "Fractile",
        "Std(u)",
        "Mean(u)",
        "t=1",
        "t=2",
        "t=3",
        "t=4",
        "t=5",
        "t=6",
    ]
)
NW_technical_inefficiency_results["Rank"] = percetile_firm_idx
NW_technical_inefficiency_results["Fractile"] = np.arange(0.05, 1, 0.1)
NW_technical_inefficiency_results["Std(u)"] = NW_std_u_hat_for_table
NW_technical_inefficiency_results["Mean(u)"] = NW_mean_u_hat_for_table
NW_technical_inefficiency_results.iloc[:, -6:] = NW_u_hat_for_table

# Compute averages
average_NW = pd.DataFrame(
    np.concatenate(
        [
            np.mean(NW_std_u_hat_for_table).reshape(-1, 1).T,
            np.mean(NW_mean_u_hat_for_table).reshape(-1, 1).T,
            np.mean(NW_u_hat_for_table, axis=0).reshape(-1, 1).T,
        ],
        axis=1,
    ),
    columns=["Std(u)", "Mean(u)", "t=1", "t=2", "t=3", "t=4", "t=5", "t=6"],
    index=["Average"],
)

print("Nadaraya Watson Scores")
display(NW_technical_inefficiency_results)
display(average_NW)

/home/zwei1/.conda/envs/d2l_f25/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Nadaraya Watson Scores


,Rank,Fractile,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
0,9,0.05,0.245780,0.403574,0.024189,0.318616,0.738029,0.688577,0.270438,0.381592
1,26,0.15,0.169354,0.239335,0.052607,0.124727,0.510406,0.431043,0.138368,0.17886
2,43,0.25,0.136559,0.232430,0.09543,0.093579,0.367388,0.457905,0.165529,0.214747
3,60,0.35,0.167328,0.237828,0.158177,0.10352,0.195613,0.605008,0.161773,0.202875
4,77,0.45,0.185032,0.224482,0.195306,0.182155,0.163092,0.627522,0.095727,0.083093
5,94,0.55,0.145018,0.196599,0.227689,0.126032,0.506266,0.104792,0.108692,0.106124
6,111,0.65,0.100930,0.287980,0.290795,0.273055,0.250433,0.501722,0.218845,0.193028
7,128,0.75,0.236154,0.332206,0.34082,0.140553,0.474093,0.773594,0.094968,0.169209
8,145,0.85,0.273121,0.407474,0.465053,0.11628,0.9142,0.542475,0.17906,0.227774
9,162,0.95,0.223640,0.543871,0.667484,0.659475,0.313284,0.918952,0.307447,0.396586


,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
Average,0.188292,0.310578,0.251755,0.213799,0.44328,0.565159,0.174085,0.215389
